# Multi-Folding Best Candidates
(using the Boltz API)

In [1]:
import pandas as pd
import os, time
from boltz_api import Boltz

In [2]:
candidates_df = pd.read_excel("../../candidates.xlsx")

## Order by lowest ipSAE Score from Proteinbase
candidates_df = candidates_df.sort_values("proteinbase_ipsae", ascending=False)

candidates_df.head()

,antibody_id,method,model_version,temperature,type,parent_antibody_id,antigen_id,note_ids,h_chain,l_chain,...,status_folded_franken_boltz2,min_ipsae_full,max_ipsae_full,min_ipsae_comp,max_ipsae_comp,submitted_to_comp,proteinbase_name,proteinbase_ipsae,on_ephrin_b2_epitope,on_ephrin_b2_epitope_comp
145,sbio-nipahgpg-148,peleke-1 + AntiBMPNN,peleke-phi-4 / 000,0.2,derivative,sbio-nipahgpg-120,nipah_gpG_compout,NaN,EVQLVESGGGLVQPGGSLRLSCAASGFNISYSSIHWVRQAPGKGLE...,DIQMTQSPSSLSASVGDRVTITCRASQSVYYAVAWYQQKPGKAPKL...,...,True,0.530209,0.607679,0.000000,0.000000,2025-11-28,rapid-otter-pearl,0.710000,False,True
117,sbio-nipahgpg-120,peleke-1 + AntiBMPNN,peleke-phi-4 / 000,0.2,derivative,sbio-nipahgpg-055,nipah_gpG_compout,NaN,EVQLVESGGGLVQPGGSLRLSCAASGFNISDSSIHWVRQAPGKGLE...,DIQMTQSPSSLSASVGDRVTITCRASQSVSSAVAWYQQKPGKAPKL...,...,True,0.000000,0.000000,0.607978,0.698886,2025-11-22,silent-wolf-ember,0.690000,False,True
52,sbio-nipahgpg-055,peleke-1,peleke-phi-4,0.2,base,NaN,nipah_gpG_compout,NaN,EVQLVESGGGLVQPGGSLRLSCAASGFNISSSSIHWVRQAPGKGLE...,DIQMTQSPSSLSASVGDRVTITCRASQSVSSAVAWYQQKPGKAPKL...,...,True,0.000000,0.004986,0.000000,0.005652,2025-11-13,frozen-goat-snow,0.630025,False,False
111,sbio-nipahgpg-114,peleke-1,peleke-phi-4,0.9,base,NaN,nipah_gpG_compout,NaN,EVQLVESGGGLVKPGGSLKVSCAASGFTFSDYSMNWVRQAPGKGLE...,DIQMTQSPSSLSASVGDRVTITCQASQDIRFYLNWYQQKPGKAPKL...,...,True,0.000000,0.000000,0.000000,0.000000,2025-11-18,brisk-vole-plume,0.611021,False,True
80,sbio-nipahgpg-083,peleke-1,peleke-phi-4,0.5,derivative,sbio-nipahgpg-063,nipah_gpG_compout,3.0,QVQLVQSGAEVKKPGSSVKVSCKASGGTFSSYTISWVRQAPGQGLE...,DIQMTQSPSSLSASVGDRVTITCRASQGISSWLAWYQQKPGKAPKL...,...,True,0.000000,0.005329,0.399019,0.516604,2025-11-13,carlet-moth-plume,0.534505,False,False


In [18]:

## nipah_gpG_comp	proteinbase (github)
# antigen_seq = "QNYTRSTDNQAVIKDALQGIQQQIKGLADKIGTEIGPKVSLIDTSSTITIPANIGLLGSKISQSTASINENVNEKCKFTLPPLKIHECNISCPNPLPFREYRPQTEGVSNLVGLPNNICLQKTSNQILKPKLISYTLPVVGQSGTCITDPLLAMDEGYFAYSHLERIGSCSRGVSKQRIIGVGEVLDRGDEVPSLFMTNVWTPPNPNTVYHCSAVYNNEFYYVLCAVSTVGDPILNSTYWSGSLMMTRLAVKPKSNGGGYNQHQLALRSIEKGRYDKVMPYGPSGIKQGDTLYFPAVGFLVRTEFKYNDSNCPITKCQYSKPENCRLSMGIRPNSHYILRSGLLKYNLSDGENPKVVFIEISDQRLSIGSPSKIYDSLGQPVFYQASFSWDTMIKFGDVLTVNPLVVNWRNNTVISRPGQSQCPRFNTCPEICWEGVYNDAFLIDRINWISAGVFLDSNQTAENPVFTVFKDNEILYRAQLASEDTNAQKTITNCFLLKNKIWCISLVEIYDTGDNVIRPKLFAVKIPEQCT"

## nipah_gpG_compout	proteinbase (output)
antigen_seq = "MPAENKKVRFENTTSDKGKIPSKVIKSYYGTMDIKKINEGLLDSKILSAFNTVIALLGSIVIIVMNIMIIQNYTRSTDNQAVIKDALQGIQQQIKGLADKIGTEIGPKVSLIDTSSTITIPANIGLLGSKISQSTASINENVNEKCKFTLPPLKIHECNISCPNPLPFREYRPQTEGVSNLVGLPNNICLQKTSNQILKPKLISYTLPVVGQSGTCITDPLLAMDEGYFAYSHLERIGSCSRGVSKQRIIGVGEVLDRGDEVPSLFMTNVWTPPNPNTVYHCSAVYNNEFYYVLCAVSTVGDPILNSTYWSGSLMMTRLAVKPKSNGGGYNQHQLALRSIEKGRYDKVMPYGPSGIKQGDTLYFPAVGFLVRTEFKYNDSNCPITKCQYSKPENCRLSMGIRPNSHYILRSGLLKYNLSDGENPKVVFIEISDQRLSIGSPSKIYDSLGQPVFYQASFSWDTMIKFGDVLTVNPLVVNWRNNTVISRPGQSQCPRFNTCPEICWEGVYNDAFLIDRINWISAGVFLDSNQTAENPVFTVFKDNEILYRAQLASEDTNAQKTITNCFLLKNKIWCISLVEIYDTGDNVIRPKLFAVKIPEQCT"


## Run Screen
Source: https://api.boltz.bio/docs/guides/protein-library-screen/

In [19]:
client = Boltz(api_key=os.environ["BOLTZ_API_KEY"])

In [20]:
target = {
    "type": "no_template",
    "entities": [
        {
            "type": "protein",
            "value": antigen_seq,
            "chain_ids": ["A"]
        }
    ]
}

In [21]:
proteins = []

## For Individual Chain Mode
for i, row in candidates_df.iterrows():
    h_chain_seq = row['h_chain']
    l_chain_seq = row['l_chain']
    antibody_id = row["antibody_id"]
    proteins.append(
        {
            "entities": [
                {"type": "protein", "value": h_chain_seq, "chain_ids": ["H"]},
                {"type": "protein", "value": l_chain_seq, "chain_ids": ["L"]}
            ], "id": antibody_id
        }
    )


In [22]:
## Submit now and download later:
screen = client.protein.library_screen.start(target=target, proteins=proteins)

In [23]:
## Print Screen ID
print(f"Screen ID: {screen.id}")

Screen ID: prot_scr_ccHKc5gt3K4DByd70Ing


In [24]:
## Poll the run for status and progress.
while screen.status not in ("succeeded", "failed", "stopped"):
    time.sleep(10)
    screen = client.protein.library_screen.retrieve(screen.id)
    p = screen.progress
    print(f"{screen.status}: {p.num_proteins_screened}/{p.total_proteins_to_screen}")


running: 0/156
running: 0/156
running: 0/156
running: 0/156
running: 0/156
running: 0/156
running: 0/156
running: 0/156
running: 0/156
running: 60/156
running: 78/156
running: 78/156
running: 78/156
running: 127/156
succeeded: 156/156


In [25]:
## Best first: highest binding confidence, then lowest interface error.
## Use external_id to correlate each result back to the protein you submitted.
results = list(client.protein.library_screen.list_results(screen.id))
results.sort(key=lambda r: (-r.metrics.binding_confidence, r.metrics.min_interaction_pae))
for r in results[:5]:
    print(
        f"{r.id}  "
        f"ext={r.external_id}  "
        f"bind={r.metrics.binding_confidence:.2f}  "
        f"structure_confidence={r.metrics.structure_confidence:.2f}  "
        f"iPAE={r.metrics.min_interaction_pae:.1f}Å  "
    )

pres_UHuk1lydKdcjvRIOS9Jv  ext=sbio-nipahgpg-058  bind=0.19  structure_confidence=0.00  iPAE=21.0Å  
pres_45jUlgDCQai0T4Q4iTsh  ext=sbio-nipahgpg-154  bind=0.15  structure_confidence=0.01  iPAE=16.7Å  
pres_tu6v5gYsKkQBqLN6Sc63  ext=sbio-nipahgpg-122  bind=0.14  structure_confidence=0.00  iPAE=21.0Å  
pres_857M5vFPrNeZt8SEgZMG  ext=sbio-nipahgpg-027  bind=0.13  structure_confidence=0.17  iPAE=6.3Å  
pres_qpy0T1KebfdLSVm3PML9  ext=sbio-nipahgpg-061  bind=0.11  structure_confidence=0.00  iPAE=20.4Å  


In [26]:
## Write results to pickle file
import pickle
with open("screen_results.pkl", "wb") as f:
    pickle.dump(results, f)

In [ ]:
# results = pickle.load(open("screen_resultss.pkl", "rb"))

In [27]:
def parse_metrics(result):
    return {
        "id": result.id,
        "external_id": result.external_id,
        "created_at": result.created_at,
        "archive_url": result.artifacts.archive.url,
        "archive_url_expires_at": result.artifacts.archive.url_expires_at,
        "structure_url": result.artifacts.structure.url,
        "structure_url_expires_at": result.artifacts.structure.url_expires_at,
        "entities": [e.to_dict() for e in result.entities],
        "binding_confidence": result.metrics.binding_confidence,
        "helix_fraction": result.metrics.helix_fraction,
        "iptm": result.metrics.iptm,
        "loop_fraction": result.metrics.loop_fraction,
        "min_interaction_pae": result.metrics.min_interaction_pae,
        "sheet_fraction": result.metrics.sheet_fraction,
        "structure_confidence": result.metrics.structure_confidence
    }

In [28]:
results_dict = [parse_metrics(results[i]) for i in range(len(results))]

In [29]:
results_df = pd.DataFrame(results_dict)

results_df.head()

,id,external_id,created_at,archive_url,archive_url_expires_at,structure_url,structure_url_expires_at,entities,binding_confidence,helix_fraction,iptm,loop_fraction,min_interaction_pae,sheet_fraction,structure_confidence
0,pres_UHuk1lydKdcjvRIOS9Jv,sbio-nipahgpg-058,2026-06-14 15:49:14.791000+00:00,https://boltz-platform-prod-compute-api-storag...,2026-06-14 16:19:30.930000+00:00,https://boltz-platform-prod-compute-api-storag...,2026-06-14 16:19:30.902000+00:00,"[{'chain_ids': ['H'], 'type': 'protein', 'valu...",0.191713,0.040179,0.259167,0.464286,20.957047,0.495536,0.002527
1,pres_45jUlgDCQai0T4Q4iTsh,sbio-nipahgpg-154,2026-06-14 15:49:14.968000+00:00,https://boltz-platform-prod-compute-api-storag...,2026-06-14 16:19:30.930000+00:00,https://boltz-platform-prod-compute-api-storag...,2026-06-14 16:19:30.902000+00:00,"[{'chain_ids': ['H'], 'type': 'protein', 'valu...",0.147972,0.039301,0.313860,0.458515,16.734669,0.502183,0.006319
2,pres_tu6v5gYsKkQBqLN6Sc63,sbio-nipahgpg-122,2026-06-14 15:49:17.061000+00:00,https://boltz-platform-prod-compute-api-storag...,2026-06-14 16:19:31.078000+00:00,https://boltz-platform-prod-compute-api-storag...,2026-06-14 16:19:31.063000+00:00,"[{'chain_ids': ['H'], 'type': 'protein', 'valu...",0.143210,0.052402,0.188670,0.458515,20.973000,0.489083,0.000907
3,pres_857M5vFPrNeZt8SEgZMG,sbio-nipahgpg-027,2026-06-14 15:49:29.082000+00:00,https://boltz-platform-prod-compute-api-storag...,2026-06-14 16:19:31.078000+00:00,https://boltz-platform-prod-compute-api-storag...,2026-06-14 16:19:31.063000+00:00,"[{'chain_ids': ['H'], 'type': 'protein', 'valu...",0.133222,0.050633,0.735356,0.434599,6.324743,0.514768,0.172001
4,pres_qpy0T1KebfdLSVm3PML9,sbio-nipahgpg-061,2026-06-14 15:48:38.776000+00:00,https://boltz-platform-prod-compute-api-storag...,2026-06-14 16:19:30.930000+00:00,https://boltz-platform-prod-compute-api-storag...,2026-06-14 16:19:30.902000+00:00,"[{'chain_ids': ['H'], 'type': 'protein', 'valu...",0.107350,0.039823,0.202073,0.446903,20.379429,0.513274,0.002292


In [30]:
results_df.to_csv("screen_results.csv", index=False)

In [ ]:
## Stop the Screen
# client.protein.library_screen.stop(screen.id)